# 3.0.0 Empirical testing of diachronic drift

This notebook computes the main empirical drift metrics for the accepted semantic-change analysis pipeline. It first evaluates category-specific drift over the semantically classified corpus and then compares those results with pooled drift over the same corpus without semantic subdivision.

We first load the aligned classified corpus and reduced mention space, compute drift metrics for labeled and pooled views of the analysis corpus, and save the resulting tables and figures for interpretation.

---

In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path


def _find_workspace_root() -> Path:
    env_root = os.environ.get("WORKSPACE_ROOT", "").strip()
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    for start in [Path.cwd(), *Path.cwd().parents]:
        if (start / "config" / "workspace.json").exists():
            return start
    raise FileNotFoundError(
        "Could not locate workspace root from WORKSPACE_ROOT or the current working directory."
    )


WORKSPACE_DIR = _find_workspace_root()
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

CODE_DIR = WORKSPACE_DIR / "code"
NOTEBOOKS_DIR = CODE_DIR / "notebooks"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
SHARED_ASSETS_DIR = paths["shared_assets"]
NOTEBOOKS_DIR = CODE_DIR / "notebooks"

print("workspace_root:", WORKSPACE_DIR)
print("shared_assets_root:", SHARED_ASSETS_DIR)

# Repo mode omits manuscript and Overleaf export surfaces.
def record_figure_output(source: Path, target_name: str | None = None) -> Path:
    return Path(source)

def record_table_output(source: Path, target_name: str | None = None) -> Path:
    return Path(source)

import gc
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dcor import energy_distance
from scipy.stats import wasserstein_distance
from sklearn.metrics.pairwise import cosine_similarity

from notebook_theme import CAT_LABELS, activate_theme, colors, style_3d_axis, style_axis, themes

FORCE_RERUN = True
THEME_NAME = "paper"
RUN_ID = "analysis_corpus"
REDUCER = "ae"
REDUCED_DIM = 200
REDUCED_LAYER_MODE = "all_mean"
REDUCED_LABEL = f"{REDUCER.upper()}-{REDUCED_DIM}_{REDUCED_LAYER_MODE}"

theme = activate_theme(THEME_NAME)

CORPUS_DIR = LOCAL_DIR / "corpora" / RUN_ID / "bins"
RUN_ROOT = OUTPUTS_DIR / "analytical-results" / RUN_ID
CORPUS_STAGE_DIR = RUN_ROOT / "corpus"
CORPUS_DATA_DIR = CORPUS_STAGE_DIR / "data"
DRIFT_DATA_DIR = RUN_ROOT / "drift" / "data"
DRIFT_PLOT_DIR = RUN_ROOT / "drift" / "plots" / THEME_NAME
CLS_PATH = CORPUS_STAGE_DIR / "corpus_classified.jsonl"
INDEX_PATH = CORPUS_DATA_DIR / "embedding_label_index.jsonl"
EMBEDDING_META_DIR = RUN_ROOT / "embeddings" / REDUCED_LAYER_MODE
REDUCTION_ROOT = OUTPUTS_DIR / "dimension-reduction" / "models" / RUN_ID
REDUCED_FILE = REDUCTION_ROOT / REDUCER / f"{REDUCER}_{REDUCED_DIM}d_{REDUCED_LAYER_MODE}.npy"

for path in [CORPUS_DATA_DIR, DRIFT_DATA_DIR, DRIFT_PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("CORPUS_DIR:", CORPUS_DIR, "exists:", CORPUS_DIR.exists())
print("RUN_ROOT:", RUN_ROOT, "exists:", RUN_ROOT.exists())
print("CLS_PATH:", CLS_PATH, "exists:", CLS_PATH.exists())
print("INDEX_PATH:", INDEX_PATH, "exists:", INDEX_PATH.exists())
print("EMBEDDING_META_DIR:", EMBEDDING_META_DIR, "exists:", EMBEDDING_META_DIR.exists())
print("REDUCED_FILE:", REDUCED_FILE, "exists:", REDUCED_FILE.exists())

if not REDUCED_FILE.exists():
    raise FileNotFoundError(f"Missing reduced embedding file: {REDUCED_FILE}")
if not CLS_PATH.exists():
    raise FileNotFoundError(f"Missing classified corpus file: {CLS_PATH}")


### Helper functions and aligned corpus loading

---

In [ ]:
# ── Shared helpers ─────────────────────────────────────────────────────────
# ── Helpers ──────────────────────────────────────────────────────────────────
def bin_start_year(label: str) -> int:
    return int(str(label).split("-")[0])

def example_key(row) -> str:
    sent = " ".join(str(row["sent"]).split())
    return f'{row["bibcode"]}|||{sent}|||{int(row["start"])}|||{int(row["end"])}'

def sample_category_examples(cat, bin_label, n=10, threshold=0.8):
    """Show up to n high-confidence examples of a category in a given bin."""
    subset = df[
        (df["bin"] == bin_label) &
        (df[f"prob_{cat}"] >= threshold)
    ].copy()

    print(f"\n=== Category {cat} | {bin_label} | prob >= {threshold} ===")
    print(f"Matches: {len(subset)}")

    if subset.empty:
        print("  No matching examples.\n")
        return

    sample_n = min(n, len(subset))
    subset = subset.sample(sample_n, random_state=42)

    for _, row in subset.iterrows():
        print(f"  [{row['year']}] {row['sent'][:500]}")
        print(
            f"  probs: A={row['prob_A']:.2f} "
            f"B={row['prob_B']:.2f} "
            f"C={row['prob_C']:.2f}\n"
        )

X_drift = REDUCED_FILE

# ── Energy Distance ────────────────────────────────────────────────────
def energy_distance_sampled(X1, X2, n_sub=4000, seed=42):
    rng = np.random.default_rng(seed)
    X1s = X1[rng.choice(len(X1), min(n_sub, len(X1)), replace=False)]
    X2s = X2[rng.choice(len(X2), min(n_sub, len(X2)), replace=False)]
    return float(energy_distance(X1s, X2s))

# ── Subspace drift ────────────────────────────────────────────────────
def subspace_drift(X1, X2, n_components=20, n_sub=8000, seed=42):
    rng = np.random.default_rng(seed)
    X1s = X1[rng.choice(len(X1), min(n_sub, len(X1)), replace=False)]
    X2s = X2[rng.choice(len(X2), min(n_sub, len(X2)), replace=False)]
    _, _, Vt1 = np.linalg.svd(X1s - X1s.mean(0), full_matrices=False)
    _, _, Vt2 = np.linalg.svd(X2s - X2s.mean(0), full_matrices=False)
    M = Vt1[:n_components] @ Vt2[:n_components].T
    sigma = np.clip(np.linalg.svd(M, compute_uv=False), -1, 1)
    return float(np.arccos(sigma).mean())

# ── Controid cosine distance ────────────────────────────────────────────────────
def centroid_cosine_distance(X1, X2):
    c1 = X1.mean(axis=0, keepdims=True)
    c2 = X2.mean(axis=0, keepdims=True)
    return float(1 - cosine_similarity(c1, c2)[0, 0])

# ── Maximum Mean Discrepancy ────────────────────────────────────────────────────
def mmd_rbf(X1, X2, n_sub=2000, sigma=None, seed=42):
    rng = np.random.default_rng(seed)
    X1s = X1[rng.choice(len(X1), min(n_sub, len(X1)), replace=False)]
    X2s = X2[rng.choice(len(X2), min(n_sub, len(X2)), replace=False)]

    if sigma is None:
        sample = np.vstack([X1s[:100], X2s[:100]])
        ix = rng.choice(len(sample), size=min(500, len(sample)), replace=False)
        sub = sample[ix]
        diffs = sub[:, None] - sub[None, :]
        dists = np.linalg.norm(diffs, axis=-1)
        sigma = np.median(dists[dists > 0])

    def rbf(A, B):
        dists_sq = np.sum((A[:, None] - B[None, :]) ** 2, axis=-1)
        return np.exp(-dists_sq / (2 * sigma ** 2))

    return float(rbf(X1s, X1s).mean() + rbf(X2s, X2s).mean() - 2 * rbf(X1s, X2s).mean())

# ── Sliced Wassesrtein Distance────────────────────────────────────────────────────
def wasserstein_1d_projected(X1, X2, n_proj=500, seed=42):
    rng = np.random.default_rng(seed)
    dim = X1.shape[1]
    directions = rng.standard_normal((n_proj, dim))
    directions /= np.linalg.norm(directions, axis=1, keepdims=True)
    P1 = X1 @ directions.T
    P2 = X2 @ directions.T
    dists = [wasserstein_distance(P1[:, j], P2[:, j]) for j in range(n_proj)]
    return float(np.mean(dists))

# ── Bootstrap ────────────────────────────────────────────────────
def bootstrap_metric(X1, X2, metric_fn, n_boot=500, seed=42):
    rng = np.random.default_rng(seed)
    scores = [
        metric_fn(
            X1[rng.choice(len(X1), len(X1), replace=True)],
            X2[rng.choice(len(X2), len(X2), replace=True)],
        )
        for _ in range(n_boot)
    ]
    return float(np.mean(scores)), [float(x) for x in np.percentile(scores, [2.5, 97.5])]


In [ ]:
# ── Reload corpus  ──────
bin_files = sorted(CORPUS_DIR.glob("*.jsonl"), key=lambda p: bin_start_year(p.stem))

base_rows = []
corpus_counts = {}
for bin_file in bin_files:
    bin_label = bin_file.stem
    with open(bin_file, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]
    corpus_counts[bin_label] = len(rows)
    for row in rows:
        if "time_bin" in row and row["time_bin"] != bin_label:
            raise ValueError(f"time_bin mismatch in {bin_label}")
        row["bin"] = bin_label
        row["example_key"] = example_key(row)
        base_rows.append(row)

df_base = pd.DataFrame(base_rows)

# ── Load classified rows ─────────────────────────────────────────────────────
with open(CLS_PATH, encoding="utf-8") as f:
    cls_rows = [json.loads(line) for line in f]

df_cls = pd.DataFrame(cls_rows)
df_cls["example_key"] = df_cls.apply(example_key, axis=1)
df_cls = df_cls.drop_duplicates(subset=["example_key"]).copy()

merge_cols = ["example_key", "prob_A", "prob_B", "prob_C", "labels"]
df = df_base.merge(
    df_cls[merge_cols],
    on="example_key",
    how="left",
    validate="one_to_one",
)

missing = df["labels"].isna().sum()
if missing:
    raise ValueError(f"Missing classifier output for {missing} corpus rows")

# ── Validate embedding metadata before loading reduced space ────────────────
meta_counts = {}
for meta_path in sorted(EMBEDDING_META_DIR.glob("*.meta.json")):
    data = json.loads(meta_path.read_text())
    bin_label = data.get("bin") or meta_path.stem.replace('.meta', '')
    rows = data.get("rows") or data.get("n_rows") or data.get("count")
    if rows is not None:
        meta_counts[str(bin_label)] = int(rows)

bin_mismatches = []
for bin_label, corpus_n in corpus_counts.items():
    meta_n = meta_counts.get(bin_label)
    if meta_n is None:
        bin_mismatches.append(f"{bin_label}: corpus={corpus_n}, embedding_meta=missing")
    elif meta_n != corpus_n:
        bin_mismatches.append(f"{bin_label}: corpus={corpus_n}, embedding_meta={meta_n}")

if bin_mismatches:
    mismatch_text = "\n  ".join(bin_mismatches)
    raise ValueError(
        "Embedding metadata does not match the current corpus bins. "
        "The embedding / dimension-reduction artifacts are stale and should be regenerated.\n  "
        f"{mismatch_text}"
    )

# ── Load reduced embeddings and verify total alignment ──────────────────────
X_reduced = np.load(REDUCED_FILE)

print("Checking gold-standard alignment:")
print(f"  corpus rows:      {len(df):>6,}")
print(f"  reduced vectors:  {X_reduced.shape[0]:>6,}")

if len(df) != X_reduced.shape[0]:
    raise ValueError(
        "Reduced embedding row count does not match the current corpus. "
        "Re-run 01b embeddings and 01c dim reduction for this corpus version.\n"
        f"Mismatch: {len(df)} corpus rows vs {X_reduced.shape[0]} reduced embeddings"
    )

print("All aligned")

# ── Derive dominant label ────────────────────────────────────────────────────
for cat in "ABC":
    df[f"has_{cat}"] = df["labels"].apply(lambda x: cat in x)

df["dominant"] = df[["prob_A", "prob_B", "prob_C"]].idxmax(axis=1).map(
    {"prob_A": "A", "prob_B": "B", "prob_C": "C"}
)
df["dominant"] = df.apply(
    lambda r: r["dominant"] if (r["has_A"] or r["has_B"] or r["has_C"]) else "none",
    axis=1,
)

print("\nDominant labels:")
print(df["dominant"].value_counts())

print("\nBin counts:")
print(df["bin"].value_counts().sort_index())

labeled_data_path = CORPUS_DATA_DIR / "labeled_data.csv"
df.to_csv(labeled_data_path, index=False)
print(f"Saved labeled data to {labeled_data_path}")


## Summary

Inspect the aligned corpus summary before the heavier drift computations begin.

---

In [ ]:
# ── Corpus summary ────────────────────────────────────────────────────
print("=== Full corpus label summary ===")
print(f"Total sentences: {len(df):,}")

print("\nPer-category counts and proportions:")
for cat in "ABC":
    n_cat = int(df[f"has_{cat}"].sum())
    pct_cat = 100 * n_cat / len(df)
    print(f"  {cat}: {n_cat:6d} ({pct_cat:5.1f}%)")

n_labels = df["labels"].apply(len)

print("\nNumber of labels per sentence:")
print(n_labels.value_counts().sort_index())

unlabeled = int((n_labels == 0).sum())
print("\nUnlabeled (no category assigned):")
print(f"  {unlabeled} ({100 * unlabeled / len(df):.1f}%)")

# ── Category x bin check ────────────────────────────────────────────────────
for cat in "ABC":
    for bin_label in ["1991-1995", "2001-2005", "2021-2025"]:
        sample_category_examples(cat, bin_label, n=5, threshold=0.8)


## Create an aligned index

Write an index JSON that keeps the corpus rows, labels, and embedding-level metadata aligned for downstream drift computation.


In [ ]:
index = []
for i, row in df.reset_index(drop=True).iterrows():
    index.append({
        "position": i,
        "example_key": row["example_key"],
        "bin": row["bin"],
        "year": int(row["year"]),
        "bibcode": row.get("bibcode"),
        "start": int(row["start"]),
        "end": int(row["end"]),
        "sent": row["sent"],
        "labels": row["labels"],
        "prob_A": float(row["prob_A"]),
        "prob_B": float(row["prob_B"]),
        "prob_C": float(row["prob_C"]),
        "has_math": bool(row.get("has_math", False)),
        "dominant": row["dominant"],
        "has_A": bool(row["has_A"]),
        "has_B": bool(row["has_B"]),
        "has_C": bool(row["has_C"]),
    })

with open(INDEX_PATH, "w", encoding="utf-8") as f:
    for row in index:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved {len(index):,} index rows to {INDEX_PATH}")

---

# Section A: Classified corpus drift

This section computes drift metrics for the semantically classified corpus.

---

### Prepare semantics

In [ ]:
# ── Prepare semantics ────────────────────────────────────────────────────
# Prepare semantics by bin and by category from the aligned reduced space
X_drift = X_reduced.astype(np.float32)

assert X_drift.shape[0] == len(df), (
    f"Mismatch: {X_drift.shape[0]} vectors vs {len(df)} records"
)

print("X_drift:", X_drift.shape)

bins = sorted(df["bin"].unique(), key=bin_start_year)

# all sentences per bin
bin_embeddings = {}
for bin_label in bins:
    mask = (df["bin"] == bin_label).to_numpy()
    bin_embeddings[bin_label] = X_drift[mask]
    print(f"{bin_label}: n={bin_embeddings[bin_label].shape[0]}")

## Single-label vs. multi-label views

### Choice parameter

Some sentences carry multiple category labels with probabilities. Choose one of the two downstream views below.

- Multi-label membership: a sentence contributes to *every label* it carries
- Exclusive assignment: a sentence contributes *only to its dominant label*

`USE_DOMINANT = False` keeps the multi-label view.

`USE_DOMINANT = True` collapses to the single most probable label.

---

In [ ]:
# ── Single vs. multi-labeled sentences ────────────────────────────────────────────────────
USE_DOMINANT = True

label_mode = "dominant" if USE_DOMINANT else "multilabel"
label_mode_title = "dominant-label" if USE_DOMINANT else "multi-label"

print(f"Label mode: {label_mode_title}")

cat_embeddings = {cat: {} for cat in "ABC"}

for bin_label in bins:
    df_bin = df[df["bin"] == bin_label]
    X_bin = X_drift[df_bin.index.to_numpy()]

    counts = {}
    for cat in "ABC":
        if USE_DOMINANT:
            mask = (df_bin["dominant"] == cat).to_numpy()
        else:
            mask = df_bin["labels"].apply(lambda x: cat in x).to_numpy()

        n_cat = int(mask.sum())
        counts[cat] = n_cat

        if n_cat < 20:
            print(f"Warning: {bin_label} {cat} only {n_cat} sentences, skipping")
            continue

        cat_embeddings[cat][bin_label] = X_bin[mask]

    print(
        f"{bin_label}: "
        + " | ".join(f"{cat}={counts[cat]}" for cat in "ABC")
    )

---

## Drift-metric run mode

### Choice parameter

Use this cell to switch between a lighter operational check and the retained full computation.

- `test` keeps the same logic but uses smaller thresholds, iteration counts, and sampling sizes
- `final` runs the retained analysis settings

---

In [ ]:
RUN = "final"  # "test" or "final"

# ── Metrics ────────────────────────────────────────────────────

RUN_CONFIG = {
    "test": {
        "n_sub_energy": 1000,
        "n_sub_mmd": 1000,
        "n_proj_sw": 100,
        "n_sub_subspace": 2000,
    },
    "final": {
        "n_sub_energy": 4000,
        "n_sub_mmd": 2000,
        "n_proj_sw": 500,
        "n_sub_subspace": 8000,
    },
}
run_cfg = RUN_CONFIG[RUN]

# ── Bootstrap ────────────────────────────────────────────────────

BOOT_CONFIG = {
    "test": {
        "n_boot_cos": 50,
        "n_boot_mmd": 50,
        "n_boot_sw": 50,
        "n_boot_ed": 25,
        "n_proj_sw_boot": 25,
        "n_sub_energy_boot": 200,
    },
    "final": {
        "n_boot_cos": 500,
        "n_boot_mmd": 500,
        "n_boot_sw": 500,
        "n_boot_ed": 200,
        "n_proj_sw_boot": 50,
        "n_sub_energy_boot": 400,
    },
}

cfg = BOOT_CONFIG[RUN]
print(f"RUN mode: {RUN}")
print(cfg)

### Metric specifications

---

In [ ]:
X_drift = X_reduced.astype(np.float32)

# ── Globals, subsamples ────────────────────────────────────────────────────
# global sigma from the chosen drift space
rng = np.random.default_rng(42)
sample_n = min(2000, len(X_drift))
sample = X_drift[rng.choice(len(X_drift), sample_n, replace=False)]
sub = sample[: min(450, len(sample))]
diffs = sub[:, None] - sub[None, :]
dists = np.linalg.norm(diffs.reshape(-1, diffs.shape[-1]), axis=-1)
GLOBAL_SIGMA = float(np.median(dists[dists > 0]))
print(f"Global sigma: {GLOBAL_SIGMA:.4f}")


### Main loop executable
---

In [ ]:
drift_path = DRIFT_DATA_DIR / f"drift_metrics_{label_mode}_{RUN}.csv"
config_path = DRIFT_DATA_DIR / f"drift_metrics_{label_mode}_{RUN}_config.json"

cache_candidates = [("requested", drift_path, config_path)]
for mode in ["final", "test"]:
    csv_path = DRIFT_DATA_DIR / f"drift_metrics_{label_mode}_{mode}.csv"
    cfg_path = DRIFT_DATA_DIR / f"drift_metrics_{label_mode}_{mode}_config.json"
    if csv_path != drift_path:
        cache_candidates.append((mode, csv_path, cfg_path))

current_config = {
    "run_mode": RUN,
    "bootstrap": cfg,
    "runtime": run_cfg,
    "use_dominant": USE_DOMINANT,
    "label_mode": label_mode,
    "theme": THEME_NAME,
    "run_id": RUN_ID,
    "reduction_file": str(REDUCED_FILE),
    "n_rows": int(len(X_drift)),
    "global_sigma": float(GLOBAL_SIGMA),
    "categories": [cat for cat in "ABC" if cat in cat_embeddings],
}

cached_config = None
cache_csv_path = None
cache_cfg_path = None

if not FORCE_RERUN:
    for mode, csv_path, cfg_path in cache_candidates:
        if csv_path.exists():
            cache_csv_path = csv_path
            cache_cfg_path = cfg_path
            break

if cache_csv_path is not None:
    df_drift = pd.read_csv(cache_csv_path)
    print(f"Loaded cached drift metrics from {cache_csv_path}")

    if cache_cfg_path.exists():
        with open(cache_cfg_path, "r", encoding="utf-8") as f:
            cached_config = json.load(f)
        print(f"Loaded cached config from {cache_cfg_path}")
    else:
        print("No cached config JSON found.")

    if cache_csv_path != drift_path:
        df_drift.to_csv(drift_path, index=False)
        print(f"Migrated cached drift metrics -> {drift_path}")
        if cached_config is not None:
            with open(config_path, "w", encoding="utf-8") as f:
                json.dump(cached_config, f, indent=2)
            print(f"Migrated cached config -> {config_path}")

    mismatches = {}
    if cached_config is not None:
        for key, expected in current_config.items():
            actual = cached_config.get(key)
            if actual != expected:
                mismatches[key] = {"cached": actual, "current": expected}

    if mismatches:
        print("Warning: cached dominant drift results were created with different settings:")
        for key, payload in mismatches.items():
            print(f"  {key}: cached={payload['cached']} | current={payload['current']}")
        print("Set FORCE_RERUN = True if you want to recompute.")
    else:
        print("Cached dominant drift settings match current configuration.")

else:
    results = []

    for cat in "ABC":
        bins_available = [b for b in bins if b in cat_embeddings[cat]]

        if len(bins_available) < 2:
            print(f"Skipping {cat}: not enough bins with data")
            continue

        X_anchor = cat_embeddings[cat][bins_available[0]]

        for i in range(len(bins_available) - 1):
            b1, b2 = bins_available[i], bins_available[i + 1]
            X1 = cat_embeddings[cat][b1]
            X2 = cat_embeddings[cat][b2]

            cos_d = centroid_cosine_distance(X1, X2)
            mmd = mmd_rbf(X1, X2, n_sub=run_cfg["n_sub_mmd"], sigma=GLOBAL_SIGMA)
            sw = wasserstein_1d_projected(X1, X2, n_proj=run_cfg["n_proj_sw"])
            ed = energy_distance_sampled(X1, X2, n_sub=run_cfg["n_sub_energy"])
            subspace = subspace_drift(X1, X2, n_sub=run_cfg["n_sub_subspace"])
            cum_cos = centroid_cosine_distance(X_anchor, X2)

            cos_mean, cos_ci = bootstrap_metric(
                X1, X2,
                centroid_cosine_distance,
                n_boot=cfg["n_boot_cos"]
            )
            mmd_mean, mmd_ci = bootstrap_metric(
                X1, X2,
                lambda a, b: mmd_rbf(a, b, sigma=GLOBAL_SIGMA),
                n_boot=cfg["n_boot_mmd"]
            )
            sw_mean, sw_ci = bootstrap_metric(
                X1, X2,
                lambda a, b: wasserstein_1d_projected(a, b, n_proj=cfg["n_proj_sw_boot"]),
                n_boot=cfg["n_boot_sw"]
            )
            ed_mean, ed_ci = bootstrap_metric(
                X1, X2,
                lambda a, b: energy_distance_sampled(a, b, n_sub=cfg["n_sub_energy_boot"]),
                n_boot=cfg["n_boot_ed"]
            )

            results.append({
                "run_mode": RUN,
                "label_mode": label_mode,
                "category": cat,
                "bin_from": b1,
                "bin_to": b2,
                "n_from": int(len(X1)),
                "n_to": int(len(X2)),
                "cosine_dist": cos_d,
                "cos_mean": cos_mean,
                "cosine_ci_low": cos_ci[0],
                "cosine_ci_high": cos_ci[1],
                "mmd": mmd,
                "mmd_mean": mmd_mean,
                "mmd_ci_low": mmd_ci[0],
                "mmd_ci_high": mmd_ci[1],
                "sliced_wasserstein": sw,
                "sw_mean": sw_mean,
                "sw_ci_low": sw_ci[0],
                "sw_ci_high": sw_ci[1],
                "energy_distance": ed,
                "ed_mean": ed_mean,
                "ed_ci_low": ed_ci[0],
                "ed_ci_high": ed_ci[1],
                "subspace_drift": subspace,
                "cumulative_cos": cum_cos,
                "use_dominant": USE_DOMINANT,
            })
            print(
                f"{cat} | {b1} -> {b2}: "
                f"cos={cos_d:.4f} [{cos_ci[0]:.4f},{cos_ci[1]:.4f}] "
                f"mmd={mmd:.4f} sw={sw:.4f} ed={ed:.4f} sub={subspace:.4f} "
                f"cum={cum_cos:.4f}"
            )

        gc.collect()

    df_drift = pd.DataFrame(results)
    df_drift.to_csv(drift_path, index=False)

    config_payload = dict(current_config)
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config_payload, f, indent=2)

    print(f"Saved config -> {config_path}")
    print(f"Saved {len(df_drift)} rows -> {drift_path}")


### Plot metrics with bootstrap CI
---

In [ ]:
# ── Transition labels ────────────────────────────────────────────────────
transitions = [
    "1986-90\n↓\n1991-95",
    "1991-95\n↓\n1996-00",
    "1996-00\n↓\n2001-05",
    "2001-05\n↓\n2006-10",
    "2006-10\n↓\n2011-15",
    "2011-15\n↓\n2016-20",
    "2016-20\n↓\n2021-25",
]

# ── Metrics to plot + labels ────────────────────────────────────────────────────
metrics = [
    ("cosine_dist",        "Centroid Cosine Distance"),
    ("mmd",                "MMD (RBF kernel)"),
    ("sliced_wasserstein", "Sliced Wasserstein Distance"),
    ("energy_distance",    "Energy Distance"),
    ("subspace_drift",     "Subspace Drift (principal angles)"),
    ("cumulative_cos",     "Cumulative Cosine Distance from baseline"),
]

plot_metric = {
    "cosine_dist":        "cos_mean",
    "mmd":                "mmd_mean",
    "sliced_wasserstein": "sw_mean",
    "energy_distance":    "ed_mean",
    "subspace_drift":     "subspace_drift",
    "cumulative_cos":     "cumulative_cos",
}

labels = {"A": "A | Referential", "B": "B | Role"} #, "C": "C | Parameter"}

# ── Plot ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
axes = axes.flatten()

ci_cols = {
    "cosine_dist":        ("cosine_ci_low",  "cosine_ci_high"),
    "mmd":                ("mmd_ci_low",     "mmd_ci_high"),
    "sliced_wasserstein": ("sw_ci_low",      "sw_ci_high"),
    "energy_distance":    ("ed_ci_low",      "ed_ci_high"),
}

for ax, (metric, title) in zip(axes, metrics):
    for cat in "AB":
        sub  = df_drift[df_drift["category"] == cat]
        vals = sub[plot_metric[metric]].values   # ← use bootstrap mean where available
        x = list(range(len(transitions)))

        ax.plot(x, vals, marker="o", linewidth=1.5, markersize=5,
                color=colors[cat], label=labels[cat])

        if metric in ci_cols:
            lo_col, hi_col = ci_cols[metric]
            if lo_col in df_drift.columns:
                ax.fill_between(
                    x,
                    sub[lo_col].values,
                    sub[hi_col].values,
                    alpha=0.15, color=colors[cat]
                )

    ax.set_title(
        title,
        fontsize=10,
        loc="center",
        bbox=dict(facecolor=theme['bg'],
                  edgecolor=theme['fg'],
                  linewidth=0.5
                  ),
        x=0.5, y=0.96,
    )
    ax.set_xticks(range(len(transitions)))
    ax.set_xticklabels(transitions, rotation=0, ha="center")
    ax.grid(alpha=0.3)

leg = fig.legend(
    handles=axes[0].get_lines()[:2], ncol=3,
    labels=list(labels.values()),
    loc="upper center", bbox_to_anchor=(0.515, 0.97), fontsize=10, markerscale=1.5,
)
frame = leg.get_frame()
frame.set_boxstyle("square,pad=0.2")
frame.set_linewidth(0.8)
frame.set_alpha(1.0)
frame.set_facecolor(theme['bg'])
frame.set_edgecolor(theme['fg'])

plt.suptitle("Semantic drift by category\ndark matter corpus (SciBERT)",
             fontsize=14, y=1.03, x=0.515, multialignment="center")
plt.tight_layout()
plt.savefig(DRIFT_PLOT_DIR / f"drift_curves_{label_mode}_{RUN}.png", dpi=300, bbox_inches="tight")
record_figure_output(DRIFT_PLOT_DIR / f"drift_curves_{label_mode}_{RUN}.png")
plt.show()

### Sliced Wasserstein
---

In [ ]:
labels = {"A": "A  | Referential", "B": "B  | Role"}

cumulative = {}
for cat in "AB":
    sub = df_drift[df_drift["category"] == cat]
    total = sub["sw_mean"].sum()
    ci_width = ((sub["sw_ci_high"] - sub["sw_ci_low"]) / 2).sum()
    cumulative[cat] = (total, ci_width)

fig, ax = plt.subplots(figsize=(12, 6))

for cat in "AB":
    sub = df_drift[df_drift["category"] == cat].reset_index(drop=True)
    x = np.arange(len(sub))

    ax.plot(x, sub["sw_mean"], marker="o", linewidth=2,
            color=colors[cat], label=labels[cat])
    ax.fill_between(x,
                    sub["sw_ci_low"],
                    sub["sw_ci_high"],
                    alpha=0.2, color=colors[cat])
ax.set_xticks(range(len(transitions)))
ax.set_xticklabels(transitions, rotation=0, ha="center", fontsize=10)
ax.set_xlabel("")
ax.set_ylabel("Sliced Wasserstein Distance")
ax.set_title("Semantic drift by category with 95% bootstrap CI\n"
             "Dark matter corpus (SciBERT, n=500 bootstrap samples)",
             loc="center", fontsize=13)
ax.grid(alpha=0.3)
box_lines = ["Aggregate SW summary\n"]
for cat in "AB":
    total, ci_width = cumulative[cat]
    box_lines.append(
        f"{labels[cat]} : total={total:.4f} | summed half-width={ci_width:.4f}"
    )
box_text = "\n".join(box_lines)

ax.text(
    0.994,
    0.985,
    box_text,
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="top",
    horizontalalignment="right",
    multialignment="left",
    family="Stix Two Math",
    color=theme['fg'],
    bbox=dict(
        pad=5,
        facecolor=theme['bg'],
        edgecolor=theme['fg'],
        alpha=1.0,
        linewidth=0.8,
    )
)
leg = ax.legend(
    loc="upper left",
    frameon=True,
    markerscale=1.1,
    facecolor=theme['bg'],
    edgecolor=theme['fg'],
    framealpha=1.0
)
frame = leg.get_frame()
frame.set_boxstyle("square,pad=0.5")
frame.set_linewidth(0.8)
plt.tight_layout()
plt.savefig(DRIFT_PLOT_DIR / f"drift_SW_bootstrap_{label_mode}_{RUN}.png", dpi=300, bbox_inches="tight")
record_figure_output(DRIFT_PLOT_DIR / f"drift_SW_bootstrap_{label_mode}_{RUN}.png")
plt.show()

### Cosine drift
---

In [ ]:
labels = {
    "A": "A | Ref",
    "B": "B | Role"
}

cumulative = {}
for cat in "AB":
    sub = df_drift[df_drift["category"] == cat]
    total = sub["cos_mean"].sum()
    ci_width = ((sub["cosine_ci_high"] - sub["cosine_ci_low"]) / 2).sum()
    cumulative[cat] = (total, ci_width)

fig, ax = plt.subplots(figsize=(12, 6))

for cat in "AB":
    sub = df_drift[df_drift["category"] == cat].reset_index(drop=True)
    x = np.arange(len(sub))

    ax.plot(
        x,
        sub["cos_mean"],
        marker="o",
        linewidth=2,
        color=colors[cat],
        label=labels[cat],
    )
    ax.fill_between(
        x,
        sub["cosine_ci_low"],
        sub["cosine_ci_high"],
        alpha=0.2,
        color=colors[cat],
    )

ax.set_xticks(range(len(transitions)))
ax.set_xticklabels(transitions, rotation=0, ha="center", fontsize=10)
ax.set_xlabel("")
ax.set_ylabel("Centroid Cosine Distance")
ax.set_title(
    "Semantic drift by category with 95% bootstrap CI\n"
    f"Dark matter corpus (SciBERT, {label_mode_title}, {REDUCED_LABEL} space)",
    loc="center",
    fontsize=13,
)
ax.grid(alpha=0.3)

box_lines = ["Aggregate cosine drift summary\n"]
for cat in "AB":
    total, ci_width = cumulative[cat]
    box_lines.append(
        f"{labels[cat]} : total={total:.4f} | summed half-width={ci_width:.4f}"
    )
box_text = "\n".join(box_lines)

ax.text(
    0.994,
    0.985,
    box_text,
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="top",
    horizontalalignment="right",
    multialignment="left",
    family="Stix Two Math",
    color=theme["fg"],
    bbox=dict(
        pad=5,
        facecolor=theme["bg"],
        edgecolor=theme["fg"],
        alpha=1.0,
        linewidth=0.8,
    ),
)

leg = ax.legend(
    loc="upper left",
    frameon=True,
    markerscale=1.1,
    facecolor=theme["bg"],
    edgecolor=theme["fg"],
    framealpha=1.0,
)

frame = leg.get_frame()
frame.set_boxstyle("square,pad=0.5")
frame.set_linewidth(0.8)

plt.tight_layout()

plot_file = DRIFT_PLOT_DIR / f"drift_bootstrap_{label_mode}_cosine_{RUN}.png"
plt.savefig(plot_file, dpi=300, bbox_inches="tight")
record_figure_output(plot_file)
print(f"Saved figure -> {plot_file}")

plt.show()


---

# Section B: Pooled corpus drift


### Drift computation
---

In [ ]:
# ── Pooled drift metrics (raw / ALL) ───────────────────────────────────────
X_drift = X_reduced.astype(np.float32)

drift_raw_path = DRIFT_DATA_DIR / f"drift_metrics_raw_{RUN}.csv"
config_raw_path = DRIFT_DATA_DIR / f"drift_metrics_raw_{RUN}_config.json"

cache_candidates = [("requested", drift_raw_path, config_raw_path)]
for mode in ["final", "test"]:
    csv_path = DRIFT_DATA_DIR / f"drift_metrics_raw_{mode}.csv"
    cfg_path = DRIFT_DATA_DIR / f"drift_metrics_raw_{mode}_config.json"
    if csv_path != drift_raw_path:
        cache_candidates.append((mode, csv_path, cfg_path))

current_config = {
    "run_mode": RUN,
    "bootstrap": cfg,
    "runtime": run_cfg,
    "category": "ALL",
    "run_id": RUN_ID,
    "reduction_file": str(REDUCED_FILE),
    "n_rows": int(len(X_drift)),
}

cached_config = None
cache_csv_path = None
cache_cfg_path = None

if not FORCE_RERUN:
    for mode, csv_path, cfg_path in cache_candidates:
        if csv_path.exists():
            cache_csv_path = csv_path
            cache_cfg_path = cfg_path
            break

if cache_csv_path is not None:
    df_drift_raw = pd.read_csv(cache_csv_path)
    print(f"Loaded cached pooled drift metrics from {cache_csv_path}")

    if cache_cfg_path.exists():
        with open(cache_cfg_path, "r", encoding="utf-8") as f:
            cached_config = json.load(f)
        print(f"Loaded cached config from {cache_cfg_path}")
    else:
        print("No cached config JSON found.")

    if cache_csv_path != drift_raw_path:
        df_drift_raw.to_csv(drift_raw_path, index=False)
        print(f"Migrated cached drift metrics -> {drift_raw_path}")
        if cached_config is not None:
            with open(config_raw_path, "w", encoding="utf-8") as f:
                json.dump(cached_config, f, indent=2)
            print(f"Migrated cached config -> {config_raw_path}")

    mismatches = {}
    if cached_config is not None:
        for key, expected in current_config.items():
            actual = cached_config.get(key)
            if actual != expected:
                mismatches[key] = {"cached": actual, "current": expected}

    if mismatches:
        print("Warning: cached pooled drift results were created with different settings:")
        for key, payload in mismatches.items():
            print(f"  {key}: cached={payload['cached']} | current={payload['current']}")
        print("Set FORCE_RERUN = True if you want to recompute.")
    else:
        print("Cached pooled drift settings match current configuration.")

    GLOBAL_SIGMA_RAW = (
        cached_config.get("global_sigma_raw")
        if cached_config is not None and cached_config.get("global_sigma_raw") is not None
        else None
    )

else:
    bin_labels_raw = sorted(df["bin"].unique(), key=bin_start_year)
    all_embeddings = {b: X_drift[(df["bin"] == b).to_numpy()] for b in bin_labels_raw}

    for b in bin_labels_raw:
        print(f"{b}: pooled n={len(all_embeddings[b])}")

    rng = np.random.default_rng(42)
    X_all = np.vstack([all_embeddings[b] for b in bin_labels_raw])
    sample = X_all[rng.choice(len(X_all), min(2000, len(X_all)), replace=False)]
    sub = sample[: min(450, len(sample))]
    diffs = sub[:, None] - sub[None, :]
    dists = np.linalg.norm(diffs.reshape(-1, diffs.shape[-1]), axis=-1)
    GLOBAL_SIGMA_RAW = float(np.median(dists[dists > 0]))
    del X_all, sample, sub, diffs, dists
    gc.collect()

    print(f"Global sigma (raw pooled): {GLOBAL_SIGMA_RAW:.4f}")

    results_raw = []
    X_anchor = all_embeddings[bin_labels_raw[0]]

    for i in range(len(bin_labels_raw) - 1):
        b1, b2 = bin_labels_raw[i], bin_labels_raw[i + 1]
        X1 = all_embeddings[b1]
        X2 = all_embeddings[b2]

        cos_d = centroid_cosine_distance(X1, X2)
        mmd = mmd_rbf(X1, X2, n_sub=run_cfg["n_sub_mmd"], sigma=GLOBAL_SIGMA_RAW)
        sw = wasserstein_1d_projected(X1, X2, n_proj=run_cfg["n_proj_sw"])
        ed = energy_distance_sampled(X1, X2, n_sub=run_cfg["n_sub_energy"])
        subspace = subspace_drift(X1, X2, n_sub=run_cfg["n_sub_subspace"])
        cum_cos = centroid_cosine_distance(X_anchor, X2)

        cos_mean, cos_ci = bootstrap_metric(X1, X2, centroid_cosine_distance, n_boot=cfg["n_boot_cos"])
        mmd_mean, mmd_ci = bootstrap_metric(X1, X2, lambda a, b: mmd_rbf(a, b, sigma=GLOBAL_SIGMA_RAW), n_boot=cfg["n_boot_mmd"])
        sw_mean, sw_ci = bootstrap_metric(X1, X2, lambda a, b: wasserstein_1d_projected(a, b, n_proj=cfg["n_proj_sw_boot"]), n_boot=cfg["n_boot_sw"])
        ed_mean, ed_ci = bootstrap_metric(X1, X2, lambda a, b: energy_distance_sampled(a, b, n_sub=cfg["n_sub_energy_boot"]), n_boot=cfg["n_boot_ed"])

        results_raw.append({
            "run_mode": RUN,
            "category": "ALL",
            "bin_from": b1,
            "bin_to": b2,
            "n_from": len(X1),
            "n_to": len(X2),
            "cosine_dist": cos_d,
            "cos_mean": cos_mean,
            "cosine_ci_low": cos_ci[0],
            "cosine_ci_high": cos_ci[1],
            "mmd": mmd,
            "mmd_mean": mmd_mean,
            "mmd_ci_low": mmd_ci[0],
            "mmd_ci_high": mmd_ci[1],
            "sliced_wasserstein": sw,
            "sw_mean": sw_mean,
            "sw_ci_low": sw_ci[0],
            "sw_ci_high": sw_ci[1],
            "energy_distance": ed,
            "ed_mean": ed_mean,
            "ed_ci_low": ed_ci[0],
            "ed_ci_high": ed_ci[1],
            "subspace_drift": subspace,
            "cumulative_cos": cum_cos,
        })

        print(
            f"ALL | {b1} -> {b2}: "
            f"cos={cos_d:.4f} [{cos_ci[0]:.4f},{cos_ci[1]:.4f}] "
            f"mmd={mmd:.4f} sw={sw:.4f} ed={ed:.4f} sub={subspace:.4f} "
            f"cum={cum_cos:.4f}"
        )

    df_drift_raw = pd.DataFrame(results_raw)
    df_drift_raw.to_csv(drift_raw_path, index=False)

    config_payload = {
        "run_mode": RUN,
        "bootstrap": cfg,
        "runtime": run_cfg,
        "category": "ALL",
        "run_id": RUN_ID,
        "reduction_file": str(REDUCED_FILE),
        "n_rows": int(len(X_drift)),
        "global_sigma_raw": float(GLOBAL_SIGMA_RAW),
    }
    with open(config_raw_path, "w", encoding="utf-8") as f:
        json.dump(config_payload, f, indent=2)

    print(f"Saved config -> {config_raw_path}")
    print(f"Saved {len(df_drift_raw)} rows -> {drift_raw_path}")

df_drift_raw


In [ ]:
gruvbox_l = ["#89b482", "#ea6962", "#e78a4e", "#a9b665", "#7daea3", "#d3869b", "#d8a657"]

transitions = [
    "1986-90\n↓\n1991-95",
    "1991-95\n↓\n1996-00",
    "1996-00\n↓\n2001-05",
    "2001-05\n↓\n2006-10",
    "2006-10\n↓\n2011-15",
    "2011-15\n↓\n2016-20",
    "2016-20\n↓\n2021-25",
]

metrics = [
    ("cosine_dist",        "Centroid Cosine Distance"),
    ("mmd",                "MMD (RBF kernel)"),
    ("sliced_wasserstein", "Sliced Wasserstein Distance"),
    ("energy_distance",    "Energy Distance"),
    ("subspace_drift",     "Subspace Drift (principal angles)"),
    ("cumulative_cos",     "Cumulative Cosine Distance from baseline"),
]

# use bootstrap means where available
plot_metric = {
    "cosine_dist":        "cos_mean",
    "mmd":                "mmd_mean",
    "sliced_wasserstein": "sw_mean",
    "energy_distance":    "ed_mean",
    "subspace_drift":     "subspace_drift",
    "cumulative_cos":     "cumulative_cos",
}

ci_cols = {
    "cosine_dist":        ("cosine_ci_low", "cosine_ci_high"),
    "mmd":                ("mmd_ci_low",    "mmd_ci_high"),
    "sliced_wasserstein": ("sw_ci_low",     "sw_ci_high"),
    "energy_distance":    ("ed_ci_low",     "ed_ci_high"),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
axes = axes.flatten()

plot_colors = gruvbox_l[0:len(metrics)]
x = list(range(len(transitions)))

legend_handles = []

for ax, (metric, title), color in zip(axes, metrics, plot_colors):
    vals = df_drift_raw[plot_metric[metric]].values

    line, = ax.plot(
        x, vals,
        marker="o",
        linewidth=1.5,
        markersize=5,
        color=color,
        label=title,
    )

    if metric in ci_cols:
        lo_col, hi_col = ci_cols[metric]
        if lo_col in df_drift_raw.columns and hi_col in df_drift_raw.columns:
            ax.fill_between(
                x,
                df_drift_raw[lo_col].values,
                df_drift_raw[hi_col].values,
                alpha=0.18,
                color=color,
            )

    if not legend_handles:
        legend_handles.append(line)


    ax.set_title(
        title,
        fontsize=10,
        loc="center",
        bbox=dict(
            facecolor=theme["bg"],
            edgecolor=theme["fg"],
            linewidth=0.5,
        ),
        x=0.5,
        y=0.925,
    )
    ax.set_xticks(range(len(transitions)))
    ax.set_xticklabels(transitions, rotation=0, ha="center")
    ax.grid(alpha=0.3)

plt.suptitle(
    "Semantic drift\ndark matter corpus (SciBERT)",
    fontsize=14,
    y=1.02,
    x=0.50,
    multialignment="center",
)

plt.tight_layout()
plot_file = DRIFT_PLOT_DIR / f"drift_metrics_raw_{RUN}.png"
fig.savefig(plot_file, dpi=300, bbox_inches="tight")
record_figure_output(plot_file)
print(f"Saved figure -> {plot_file}")
plt.show()

### Aggregate cosine drift
---

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

sub = df_drift_raw.reset_index(drop=True)
x = np.arange(len(sub))

line_color = colors["A"] if "A" in colors else "#2a9d8f"

ax.plot(
    x,
    sub["cos_mean"],
    marker="o",
    linewidth=2,
    color=line_color,
    label="ALL | pooled corpus",
)

ax.fill_between(
    x,
    sub["cosine_ci_low"],
    sub["cosine_ci_high"],
    alpha=0.2,
    color=line_color,
)

ax.axvline(
    x=3,
    color=theme["fg"],
    linestyle="--",
    alpha=0.75,
    linewidth=1,
)


ax.set_xticks(range(len(transitions)))
ax.set_xticklabels(transitions, rotation=0, ha="center", fontsize=10)
ax.set_xlabel("")
ax.set_ylabel("Centroid Cosine Distance")
ax.set_title(
    "Pooled semantic drift with 95% bootstrap CI\n"
    f"Dark matter corpus (SciBERT, pooled {REDUCED_LABEL} space)",
    loc="center",
    fontsize=13,
)
ax.grid(alpha=0.3)

total = sub["cos_mean"].sum()
ci_width = ((sub["cosine_ci_high"] - sub["cosine_ci_low"]) / 2).sum()

box_text = "\n".join([
    "Aggregate cosine drift summary",
    "",
    f"ALL |\nPooled corpus : total={total:.4f}\nSummed half-width={ci_width:.4f}",
])
ax.text(
    0.006,
    0.85,
    box_text,
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="top",
    horizontalalignment="left",
    multialignment="left",
    family="Stix Two Math",
    color=theme["fg"],
    bbox=dict(
        pad=5,
        facecolor=theme["bg"],
        edgecolor=theme["fg"],
        alpha=1.0,
        linewidth=0.8,
    ),
)

leg = ax.legend(
    loc="upper left",
    frameon=True,
    markerscale=1.1,
    facecolor=theme["bg"],
    edgecolor=theme["fg"],
    framealpha=1.0,
)

frame = leg.get_frame()
frame.set_boxstyle("square,pad=0.5")
frame.set_linewidth(0.8)

plt.tight_layout()

plot_file = DRIFT_PLOT_DIR / f"drift_bootstrap_raw_cosine_{RUN}.png"
plt.savefig(plot_file, dpi=300, bbox_inches="tight")
record_figure_output(plot_file)
print(f"Saved figure -> {plot_file}")

plt.show()


### Sliced Wasserstein drift
---

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

sub = df_drift_raw.reset_index(drop=True)
x = np.arange(len(sub))

line_color = colors["A"] if "A" in colors else "#2a9d8f"

ax.plot(
    x,
    sub["sw_mean"],
    marker="o",
    linewidth=2,
    color=line_color,
    label="ALL | pooled corpus",
)

ax.fill_between(
    x,
    sub["sw_ci_low"],
    sub["sw_ci_high"],
    alpha=0.2,
    color=line_color,
)

ax.axvline(
    x=3,
    color=theme["fg"],
    linestyle="--",
    alpha=0.75,
    linewidth=1,
)

ax.set_xticks(range(len(transitions)))
ax.set_xticklabels(transitions, rotation=0, ha="center", fontsize=10)
ax.set_xlabel("")
ax.set_ylabel("Sliced Wasserstein Distance")
ax.set_title(
    "Pooled semantic drift with 95% bootstrap CI\n"
    f"Dark matter corpus (SciBERT, pooled {REDUCED_LABEL} space)",
    loc="center",
    fontsize=13,
)
ax.grid(alpha=0.3)

total = sub["sw_mean"].sum()
ci_width = ((sub["sw_ci_high"] - sub["sw_ci_low"]) / 2).sum()

box_text = "\n".join([
    "Aggregate SW summary",
    "",
    f"ALL |\nPooled corpus : total={total:.4f}\nSummed half-width={ci_width:.4f}",
])

ax.text(
    0.994,
    0.87,
    box_text,
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="top",
    horizontalalignment="right",
    multialignment="left",
    family="Stix Two Math",
    color=theme["fg"],
    bbox=dict(
        pad=5,
        facecolor=theme["bg"],
        edgecolor=theme["fg"],
        alpha=1.0,
        linewidth=0.8,
    ),
)

leg = ax.legend(
    loc="upper right",
    frameon=True,
    markerscale=1.1,
    facecolor=theme["bg"],
    edgecolor=theme["fg"],
    framealpha=1.0,
)

frame = leg.get_frame()
frame.set_boxstyle("square,pad=0.5")
frame.set_linewidth(0.8)

plt.tight_layout()

plot_file = DRIFT_PLOT_DIR / f"drift_bootstrap_raw_sw_{RUN}.png"
plt.savefig(plot_file, dpi=300, bbox_inches="tight")
record_figure_output(plot_file)
print(f"Saved figure -> {plot_file}")

plt.show()
